In [17]:
%matplotlib inline

from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.linear_model import LinearRegression

from ultralytics import YOLO
from PIL import Image

import matplotlib.pyplot as plt
import mediapipe as mp
import cv2
import time
import numpy as np
import pandas as pd
import os
import shutil


import warnings
warnings.filterwarnings('ignore')

In [18]:
# Загрузка модели
model = YOLO('yolo11n-pose.pt')  # load an official model

points = 17
path_train = "DATASET/TRAIN/" # Путь к датасету
path_test = "DATASET/TEST/" # Путь к датасету
path_RESULTS = "Results/" # Путь к папке с результурирующими изображениями
path_ERR = "Results/ERR/" # Путь к папке с неверно классифицированными изображениями

In [19]:
Key_Points_YOLO = ['Nose', 'LeftEye', 'RightEye', 'LeftEar', 'RightEar', 
                   'LeftShoulder', 'RightShoulder', 'LeftElbow', 'RightElbow', 
                   'LeftWrist', 'RightWrist', 'LeftHip', 'RightHip', 
                   'LeftKnee', 'RightKnee', 'LeftAnkle', 'RightAnkle'
                  ]
classes = ['downdog', 'goddess', 'plank', 'tree', 'warrior2']

# Создаем словарь для замены
mapping = {'downdog': 0, 'goddess': 1, 'plank': 2, 'tree': 3, 'warrior2': 4}


In [20]:
# Определяем пути к папкам
err_folder = os.path.join(os.getcwd(), path_RESULTS)
results_folder = os.path.join(os.getcwd(), path_ERR)

# Удаляем папки, если они существуют
if os.path.exists(results_folder):
    shutil.rmtree(results_folder)
    print(f"Папка {results_folder} удалена")

if os.path.exists(err_folder):
    shutil.rmtree(err_folder)
    print(f"Папка {err_folder} удалена")

# Создаем папки заново
os.makedirs(results_folder, exist_ok=True)
print(f"Папка {err_folder} создана")
print(f"Папка {results_folder} создана")

Папка /home/demon/Projects/cours_pr/Results/ создана
Папка /home/demon/Projects/cours_pr/Results/ERR/ создана


In [21]:
# Загрузка и создание обучающего набора
data_train = pd.read_csv("dataset_train_yolo.csv")
# Заменяем значения
data_train['target'] = data_train['target'].map(mapping)

X_yolo = data_train.iloc[:, 2:-1]
Y_yolo = data_train['target']

# Загрузка и создание тестового набора
data_test = pd.read_csv("dataset_test_yolo.csv")
# Заменяем значения
data_test['target'] = data_test['target'].map(mapping)

X_test_yolo = data_test.iloc[:, 2:-1]
Y_test_yolo = data_test['target']

# Задаём гиперпараметры
params = {
    'n_estimators': 100,      # Количество деревьев
    'max_depth': 10,           # Максимальная глубина дерева
    'min_samples_split': 5,   # Минимальное число образцов для разделения узла
    'min_samples_leaf': 1,    # Минимальное число образцов в листе
    'random_state': 42,       # Для воспроизводимости
}

# Создаём модель
best_model = RandomForestClassifier(**params)

# Обучаем модель
best_model.fit(X_yolo, Y_yolo)

count = 0
path = path_test
image_err = []
time_inference = 0
time_proc = 0
time_cl = 0
for dr in os.listdir(path): # Перебор папок с видами поз
    for image in os.listdir(path + "/" +dr): # Перебор файлов в каждой папке
        count +=1
        start_file = time.time()
        name_file = path + "/" + dr + "/" + image
        start_time = time.time()
        #data = preprocessing(name_file)
                
        # Загрузка выбранного файла
        #path ='DATASET/Train/goddess/00000137.jpg'# 'DATASET/Test/plank/00000015.jpg'             #'DATASET/Test/warrior2/00000093.jpg' 'DATASET/Train/goddess/00000137.jpg'
        temp = []
        img = cv2.imread(name_file)
        # Копирование и конвертация изображения в RGB
        imageWidth, imageHeight = img.shape[:2]
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Преобразование BGR модели OpenCV в RGB модель, с которой работает YOLO
        
        # Обнаружение позы            
        results = model(imgRGB)  # predict on an image 
        
        # Построение скелетной модели
        # Извлечение результатов
        for result in results[0]:
            xy = result.keypoints.xy  # x and y coordinates
            xyn = result.keypoints.xyn  # normalized
            kpts = result.keypoints.data  # x, y, visibility (if available)
        keypoints = np.array(xy[0])
        normal_keypoints = np.array(xyn[0])
        for i in range(len(normal_keypoints)):
            temp = temp + [normal_keypoints[i][0], normal_keypoints[i][1]] # Добавление ключевых точек
            
        print('Время препроцессинга: ', time.time()-start_time)
        time_proc += time.time()-start_time
        
        # Предсказание построенным классификатором класса позы по построенной скелетной модели    
        start_time = time.time()
        res = best_model.predict([temp])
        print('Время определения класса: ', time.time()-start_time) 
        time_cl += time.time()-start_time
        
        label_predict = res[0]

        # Вывод результата
        print('Файл - ', name_file)
        print('Эталонное название позы - ', name_file.split('/')[3]) 
        print('Предсказанное название позы - ', classes[label_predict]) 
        
        time_vis = time.time()
        time_file = time_vis - start_file

        
        if name_file.split('/')[3] != classes[label_predict]:
            image_err.append(name_file)

        # Визуализация результата
        for i, r in enumerate(results):
            # Plot results image
            im_rgb = r.plot(conf = False, kpt_radius = 7, boxes = False, masks = False)  # BGR-order numpy array
           
            # Преобразуем в PIL Image
            pil_im = Image.fromarray(im_rgb)
            #r.show()
            pil_im.save(path_RESULTS+dr+image)
            if name_file.split('/')[3] != classes[label_predict]:
                pil_im.save(path_ERR+dr+'_'+str(classes[label_predict])+'_'+image)
           
        time_inference += time_file
    
print('Неверно определен класс для ',len(image_err), ' изображений:\n')
print('\n'.join(image_err)) 
    
print('Общее предобработки (построения скелетной модели) и классификации каждого изображения: ', time_inference)
print('Среднее время предобработки одного изображения: ', time_proc/count)
print('Среднее время определения класса для одного изображения:', time_cl/count)


0: 320x640 1 person, 72.4ms
Speed: 2.4ms preprocess, 72.4ms inference, 0.7ms postprocess per image at shape (1, 3, 320, 640)
Время препроцессинга:  0.15422654151916504
Время определения класса:  0.0059397220611572266
Файл -  DATASET/TEST//goddess/00000016.jpg
Эталонное название позы -  goddess
Предсказанное название позы -  tree

0: 640x640 1 person, 86.0ms
Speed: 2.6ms preprocess, 86.0ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
Время препроцессинга:  0.1114342212677002
Время определения класса:  0.006679534912109375
Файл -  DATASET/TEST//goddess/00000057.jpg
Эталонное название позы -  goddess
Предсказанное название позы -  goddess

0: 640x640 1 person, 72.2ms
Speed: 2.7ms preprocess, 72.2ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
Время препроцессинга:  0.08810734748840332
Время определения класса:  0.006131410598754883
Файл -  DATASET/TEST//goddess/00000036.jpg
Эталонное название позы -  goddess
Предсказанное название позы -  goddess



libpng warning: iCCP: known incorrect sRGB profile


0: 448x640 1 person, 79.1ms
Speed: 1.6ms preprocess, 79.1ms inference, 0.8ms postprocess per image at shape (1, 3, 448, 640)
Время препроцессинга:  0.09170842170715332
Время определения класса:  0.005994319915771484
Файл -  DATASET/TEST//goddess/00000095.jpeg
Эталонное название позы -  goddess
Предсказанное название позы -  warrior2

0: 384x640 1 person, 78.5ms
Speed: 2.5ms preprocess, 78.5ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)
Время препроцессинга:  0.08606672286987305
Время определения класса:  0.006576061248779297
Файл -  DATASET/TEST//goddess/00000024.jpg
Эталонное название позы -  goddess
Предсказанное название позы -  goddess

0: 512x640 1 person, 83.9ms
Speed: 2.2ms preprocess, 83.9ms inference, 0.9ms postprocess per image at shape (1, 3, 512, 640)
Время препроцессинга:  0.09582972526550293
Время определения класса:  0.006231546401977539
Файл -  DATASET/TEST//goddess/00000020.png
Эталонное название позы -  goddess
Предсказанное название позы -  godde

libpng warning: iCCP: known incorrect sRGB profile



0: 480x640 1 person, 81.9ms
Speed: 2.2ms preprocess, 81.9ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)
Время препроцессинга:  0.43560099601745605
Время определения класса:  0.0061473846435546875
Файл -  DATASET/TEST//tree/00000047.png
Эталонное название позы -  tree
Предсказанное название позы -  tree

0: 640x480 1 person, 60.5ms
Speed: 1.7ms preprocess, 60.5ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 480)
Время препроцессинга:  0.07552051544189453
Время определения класса:  0.0060694217681884766
Файл -  DATASET/TEST//tree/00000024.jpg
Эталонное название позы -  tree
Предсказанное название позы -  tree

0: 640x544 1 person, 102.6ms
Speed: 2.3ms preprocess, 102.6ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 544)
Время препроцессинга:  0.10992717742919922
Время определения класса:  0.00672149658203125
Файл -  DATASET/TEST//tree/00000061.jpg
Эталонное название позы -  tree
Предсказанное название позы -  tree

0: 448x640 1 person, 

libpng warning: iCCP: known incorrect sRGB profile



0: 480x640 1 person, 61.6ms
Speed: 2.4ms preprocess, 61.6ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)
Время препроцессинга:  0.4101278781890869
Время определения класса:  0.006185293197631836
Файл -  DATASET/TEST//tree/00000019.png
Эталонное название позы -  tree
Предсказанное название позы -  tree

0: 640x416 1 person, 76.1ms
Speed: 2.0ms preprocess, 76.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 416)
Время препроцессинга:  0.12006640434265137
Время определения класса:  0.006494760513305664
Файл -  DATASET/TEST//tree/00000039.jpg
Эталонное название позы -  tree
Предсказанное название позы -  warrior2

0: 448x640 1 person, 61.9ms
Speed: 1.9ms preprocess, 61.9ms inference, 1.0ms postprocess per image at shape (1, 3, 448, 640)
Время препроцессинга:  0.07805776596069336
Время определения класса:  0.006914854049682617
Файл -  DATASET/TEST//tree/00000033.jpg
Эталонное название позы -  tree
Предсказанное название позы -  tree

0: 640x448 1 person, 